# UTS Sistem Cerdas
**Kasus:** Deteksi dini risiko stunting pada balita Puskesmas Harapan Sehat  
**Nama:** M Zainur Rahman Al Faiqi | **NIM:** 2411003 | **Prodi:** Informatika 24

Notebook ini menggunakan Dataset A dan Dataset B (masing-masing 200 baris) serta ketentuan Lampiran Data Pendukung. Data adalah data sintetis untuk pembelajaran, bukan rekam medis.

## Ketentuan dan asumsi analisis

Notebook ini menjawab studi kasus deteksi dini risiko stunting balita di Puskesmas Harapan Sehat dengan alur berikut:
1. Pemeriksaan kualitas data (data kosong, duplikat, kategori, rentang umur).
2. Perhitungan Z-score BB/U dan TB/U sebagai estimasi pendekatan.
3. FIS Mamdani untuk skor risiko (Soal 1a).
4. Sistem Pakar berbasis rule IF–THEN (Soal 1b).
5. Evaluasi XGBoost, Decision Tree, FIS, dan Sistem Pakar pada data uji yang sama (Soal 1c).
6. Forward chaining dan backward chaining pada kasus Hana (Soal 2).

**Ketentuan yang diikuti**
- **Data:** Dataset A (FIS dan Sistem Pakar) dan Dataset B (klasifikasi), masing-masing 200 baris dengan subjek dan ID yang sama. Seluruhnya data sintetis untuk pembelajaran, bukan rekam medis. Analisis hanya memakai kedua dataset ini.
- **XGBoost:** target `Label_risiko`; fitur umur, BB, TB, LILA, jenis kelamin, dan ASI eksklusif. `ID` dan `Nama` tidak dipakai sebagai fitur.
- **Evaluasi:** split 80:20 stratified dengan `random_state=42`; tuning lima hyperparameter dengan Stratified 5-Fold CV hanya pada data latih; accuracy, precision, recall, F1-score, dan ROC-AUC dilaporkan pada data uji.
- **Z-score:** dihitung sebagai estimasi pendekatan (bukan standar WHO) hanya dari Dataset A dan B; penjelasan dan rumusnya ada pada bagian berikutnya.
- **Berat lahir** tidak tersedia pada dataset sehingga tidak dipakai sebagai syarat rule.

In [ ]:
# 1. Import library
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# 2. Upload dataset jika belum tersedia
file_a = Path("Dataset_A_FIS_SistemPakar_200.csv")
file_b = Path("Dataset_B_XGBoost_200.csv")

if not file_a.exists() or not file_b.exists():
    from google.colab import files
    uploaded = files.upload()

    # Sesuaikan dengan nama file yang benar-benar diunggah
    for nama in uploaded.keys():
        if "Dataset_A" in nama:
            file_a = Path(nama)
        elif "Dataset_B" in nama:
            file_b = Path(nama)

# 3. Periksa keberadaan file
assert file_a.exists(), "Dataset A belum ditemukan!"
assert file_b.exists(), "Dataset B belum ditemukan!"

# 4. Baca dataset
df_a = pd.read_csv(file_a)
df_b = pd.read_csv(file_b)

# 5. Tampilkan informasi dataset
print("Dataset A:", df_a.shape)
print("Dataset B:", df_b.shape)

display(df_a.head())
display(df_b.head())


Saving Dataset_A_FIS_SistemPakar_200.csv to Dataset_A_FIS_SistemPakar_200.csv
Saving Dataset_B_XGBoost_200.csv to Dataset_B_XGBoost_200.csv
Dataset A: (200, 9)
Dataset B: (200, 9)


,ID,Nama,Umur_bulan,BB_kg,TB_cm,LILA_cm,Jenis_kelamin,ASI_eksklusif_6_bulan,Status_gizi
0,B01,Andi Pratama,24,10.2,82.0,13.5,Laki-laki,Ya,Normal
1,B02,Siti Rahayu,36,11.8,88.0,12.1,Perempuan,Tidak,Gizi Kurang
2,B03,Budi Santoso,18,7.4,74.0,11.0,Laki-laki,Tidak,Stunting
3,B04,Lena Wulandari,48,15.1,96.0,14.2,Perempuan,Ya,Normal
4,B05,Rizky Aditya,12,6.8,68.0,10.5,Laki-laki,Tidak,Stunting


,ID,Nama,Umur_bulan,BB_kg,TB_cm,LILA_cm,Jenis_kelamin,ASI_eksklusif_6_bulan,Label_risiko
0,B01,Andi Pratama,24,10.2,82.0,13.5,Laki-laki,Ya,0 (Normal)
1,B02,Siti Rahayu,36,11.8,88.0,12.1,Perempuan,Tidak,1 (Risiko)
2,B03,Budi Santoso,18,7.4,74.0,11.0,Laki-laki,Tidak,1 (Risiko)
3,B04,Lena Wulandari,48,15.1,96.0,14.2,Perempuan,Ya,0 (Normal)
4,B05,Rizky Aditya,12,6.8,68.0,10.5,Laki-laki,Tidak,1 (Risiko)


In [ ]:

# Pemeriksaan kualitas Dataset A dan Dataset B

print("=== DATASET A ===")
print("Kolom:", df_a.columns.tolist())
print("\nData kosong:")
print(df_a.isnull().sum())
print("\nData duplikat:", df_a.duplicated().sum())

print("\n=== DATASET B ===")
print("Kolom:", df_b.columns.tolist())
print("\nData kosong:")
print(df_b.isnull().sum())
print("\nData duplikat:", df_b.duplicated().sum())

print("\n=== DISTRIBUSI LABEL DATASET A ===")
print(df_a["Status_gizi"].value_counts())

print("\n=== DISTRIBUSI LABEL DATASET B ===")
print(df_b["Label_risiko"].value_counts())


=== DATASET A ===
Kolom: ['ID', 'Nama', 'Umur_bulan', 'BB_kg', 'TB_cm', 'LILA_cm', 'Jenis_kelamin', 'ASI_eksklusif_6_bulan', 'Status_gizi']

Data kosong:
ID                       0
Nama                     0
Umur_bulan               0
BB_kg                    0
TB_cm                    0
LILA_cm                  0
Jenis_kelamin            0
ASI_eksklusif_6_bulan    0
Status_gizi              0
dtype: int64

Data duplikat: 0

=== DATASET B ===
Kolom: ['ID', 'Nama', 'Umur_bulan', 'BB_kg', 'TB_cm', 'LILA_cm', 'Jenis_kelamin', 'ASI_eksklusif_6_bulan', 'Label_risiko']

Data kosong:
ID                       0
Nama                     0
Umur_bulan               0
BB_kg                    0
TB_cm                    0
LILA_cm                  0
Jenis_kelamin            0
ASI_eksklusif_6_bulan    0
Label_risiko             0
dtype: int64

Data duplikat: 0

=== DISTRIBUSI LABEL DATASET A ===
Status_gizi
Normal         124
Gizi Kurang     41
Stunting        35
Name: count, dtype: int64

=== DISTRI

In [ ]:
# Memeriksa kategori pada kolom penting

print("Jenis kelamin Dataset A:")
print(df_a["Jenis_kelamin"].value_counts())

print("\nASI eksklusif Dataset A:")
print(df_a["ASI_eksklusif_6_bulan"].value_counts())

print("\nJenis kelamin Dataset B:")
print(df_b["Jenis_kelamin"].value_counts())

print("\nASI eksklusif Dataset B:")
print(df_b["ASI_eksklusif_6_bulan"].value_counts())

Jenis kelamin Dataset A:
Jenis_kelamin
Laki-laki    101
Perempuan     99
Name: count, dtype: int64

ASI eksklusif Dataset A:
ASI_eksklusif_6_bulan
Ya       112
Tidak     88
Name: count, dtype: int64

Jenis kelamin Dataset B:
Jenis_kelamin
Laki-laki    101
Perempuan     99
Name: count, dtype: int64

ASI eksklusif Dataset B:
ASI_eksklusif_6_bulan
Ya       112
Tidak     88
Name: count, dtype: int64


In [ ]:
print("Rentang umur Dataset A:")
print("Umur terkecil:", df_a["Umur_bulan"].min(), "bulan")
print("Umur terbesar:", df_a["Umur_bulan"].max(), "bulan")

print("\nJumlah balita berdasarkan kelompok umur:")
print("Usia 0–23 bulan:", df_a["Umur_bulan"].between(0, 23).sum())
print("Usia 24–59 bulan:", df_a["Umur_bulan"].between(24, 59).sum())

Rentang umur Dataset A:
Umur terkecil: 0 bulan
Umur terbesar: 59 bulan

Jumlah balita berdasarkan kelompok umur:
Usia 0–23 bulan: 94
Usia 24–59 bulan: 106


In [ ]:
print("Tipe data umur:", df_a["Umur_bulan"].dtype)

print("\nJumlah data tiap umur:")
print(df_a["Umur_bulan"].value_counts().sort_index())

Tipe data umur: int64

Jumlah data tiap umur:
Umur_bulan
0     4
1     4
2     1
3     1
4     5
5     3
6     4
7     2
8     5
9     6
10    3
11    6
12    7
13    5
14    3
15    4
16    2
17    2
18    4
19    3
20    6
21    9
22    4
23    1
24    2
25    4
26    4
27    4
28    3
29    2
30    1
31    3
32    7
33    2
34    2
35    2
36    4
37    5
38    4
39    4
40    4
41    4
42    3
43    2
44    2
45    2
46    3
47    2
48    2
49    1
50    2
51    2
52    4
53    1
54    4
55    5
56    4
57    1
58    3
59    2
Name: count, dtype: int64


## Perhitungan Z-score (estimasi pendekatan)

Soal 1a dan 1b memerlukan Z-score BB/U (berat badan menurut umur) dan TB/U (tinggi badan menurut umur). Tabel standar WHO tidak termasuk data yang disediakan (hanya Dataset A dan B), sehingga Z-score dihitung sebagai **estimasi pendekatan**. Z-score menyatakan berapa simpangan baku sebuah nilai berada di atas atau di bawah nilai tengah kelompok acuannya:

$$Z=\frac{x-\text{median}}{SD}$$

dengan $x$ adalah BB (kg) untuk Z-BB/U atau TB (cm) untuk Z-TB/U. Nilai Z negatif berarti di bawah median kelompok acuan; semakin negatif, semakin jauh di bawahnya.

### Contoh B03 (Budi Santoso)
Data: umur 18 bulan, BB 7,4 kg, TB 74 cm. Median dan SD yang diberikan pada soal: BB 10,9 ± 1,2 kg dan TB 82,3 ± 3,2 cm.

$$Z_{BB/U}=\frac{7{,}4-10{,}9}{1{,}2}=-2{,}92 \qquad Z_{TB/U}=\frac{74-82{,}3}{3{,}2}=-2{,}59$$

### 200 balita pada dataset
Median dan SD dihitung dari Dataset A untuk setiap kombinasi **jenis kelamin** dan **kelompok umur** (0–11, 12–23, 24–35, 36–47, 48–59 bulan), tanpa memakai kolom label. SD yang dipakai adalah SD sampel:

$$SD_g=\sqrt{\frac{\sum_{i=1}^{n_g}\left(x_i-\bar{x}_g\right)^2}{n_g-1}}$$

dengan $g$ adalah kelompok (jenis kelamin dan kelompok umur) dan $n_g$ jumlah balita pada kelompok tersebut. Dataset B berisi subjek yang sama dengan Dataset A, sehingga memakai tabel referensi yang sama.

### Kategori berdasarkan Z-score
- **BB/U:** Z < −3 berat badan sangat kurang; −3 ≤ Z < −2 berat badan kurang; −2 ≤ Z ≤ 2 normal; Z > 2 berat badan lebih.
- **TB/U:** Z < −3 sangat pendek; −3 ≤ Z < −2 pendek; −2 ≤ Z ≤ 3 normal; Z > 3 tinggi.

### Cara membaca hasilnya
1. Nilai Z menunjukkan posisi balita dibandingkan teman sebaya dalam dataset ini, bukan terhadap standar WHO. Hasilnya adalah estimasi pembelajaran dan bukan diagnosis.
2. Kelompok yang kecil membuat median dan SD kurang stabil; jumlah data (n) setiap kelompok ditampilkan pada tabel referensi di bawah.
3. Posisi pengukuran (telentang atau berdiri) tidak tersedia pada dataset sehingga tidak dikoreksi.

In [ ]:
import numpy as np
import pandas as pd

# ---------- Referensi kohort: median & SD per jenis kelamin dan kelompok umur ----------
BATAS_UMUR = [-1, 11, 23, 35, 47, 60]
LABEL_UMUR = ["0-11", "12-23", "24-35", "36-47", "48-59"]

def kelompok_umur(umur):
    return pd.cut(umur, bins=BATAS_UMUR, labels=LABEL_UMUR).astype(str)

def buat_referensi(df):
    d = df.assign(Kelompok_umur=kelompok_umur(df["Umur_bulan"]))
    return (d.groupby(["Jenis_kelamin", "Kelompok_umur"])
              .agg(n=("ID", "count"),
                   BB_median=("BB_kg", "median"), BB_sd=("BB_kg", "std"),
                   TB_median=("TB_cm", "median"), TB_sd=("TB_cm", "std")))

def hitung_z_kohort(df, ref):
    """Z = (x - median) / SD, memakai referensi kohort (estimasi pembelajaran, BUKAN WHO)."""
    kunci = pd.MultiIndex.from_arrays([df["Jenis_kelamin"].to_numpy(),
                                       kelompok_umur(df["Umur_bulan"]).to_numpy()])
    r = ref.reindex(kunci)
    z_bb = (df["BB_kg"].to_numpy() - r["BB_median"].to_numpy()) / r["BB_sd"].to_numpy()
    z_tb = (df["TB_cm"].to_numpy() - r["TB_median"].to_numpy()) / r["TB_sd"].to_numpy()
    return pd.Series(z_bb, index=df.index), pd.Series(z_tb, index=df.index)

ref_kohort = buat_referensi(df_a)
print("Tabel referensi kohort (dari Dataset A):")
display(ref_kohort.round(2))
kecil = ref_kohort[ref_kohort["n"] < 8]
if len(kecil):
    print("Peringatan: kelompok dengan n < 8 (estimasi kurang stabil):", list(kecil.index))

df_a["Z_BB_U"], df_a["Z_TB_U"] = hitung_z_kohort(df_a, ref_kohort)
print("Z-score kosong -> BB/U:", df_a["Z_BB_U"].isna().sum(), "| TB/U:", df_a["Z_TB_U"].isna().sum())
display(df_a[["ID", "Nama", "Umur_bulan", "Jenis_kelamin", "BB_kg", "TB_cm", "Z_BB_U", "Z_TB_U"]].head(10))
display(df_a[["Z_BB_U", "Z_TB_U"]].describe().round(2))

### Pemeriksaan Z-score kohort terhadap label dataset
Z-score diperiksa terhadap `Status_gizi` untuk melihat apakah estimasi ini membedakan kelompok. Idealnya rata-rata Z-score kelompok *Normal* lebih tinggi daripada *Gizi Kurang* dan *Stunting*. Hasil pemeriksaan ini dipakai untuk membaca kinerja FIS dan Sistem Pakar secara jujur.

In [ ]:
ringkas = (df_a.groupby("Status_gizi")[["Z_BB_U", "Z_TB_U"]]
           .agg(["count", "mean", "median"]).round(2))
display(ringkas)

print("Proporsi balita dengan Z < -2 per kelompok label:")
display(pd.DataFrame({
    "BB/U < -2": df_a.groupby("Status_gizi")["Z_BB_U"].apply(lambda s: (s < -2).mean()),
    "TB/U < -2": df_a.groupby("Status_gizi")["Z_TB_U"].apply(lambda s: (s < -2).mean()),
}).round(2))

## Soal 1a — FIS Mamdani

### Estimasi Z-score contoh B03 (Budi Santoso)
Data contoh: umur 18 bulan, BB 7,4 kg, TB 74 cm. Parameter pada soal: median BB/U 10,9 kg dan SD 1,2 kg; median TB/U 82,3 cm dan SD 3,2 cm.

$$Z_{BB/U}=\frac{7{,}4-10{,}9}{1{,}2}=-2{,}92 \qquad Z_{TB/U}=\frac{74-82{,}3}{3{,}2}=-2{,}59$$

Angka ini adalah **estimasi pendekatan** berdasarkan median/SD contoh pada soal. Sebagai pembanding, notebook juga menampilkan Z-score B03 hasil estimasi kohort (Dataset A). Estimasi soal dipakai untuk menjawab contoh B03, sedangkan Z-score kohort dipakai untuk memproses 200 balita.

### Rancangan variabel fuzzy
| Variabel | Himpunan fuzzy | Fungsi keanggotaan |
|---|---|---|
| Z-BB/U | rendah / normal / tinggi | trapesium (−∞, −2,5, −1,5) / segitiga (−2,5, 0, 2,5) / trapesium (1,5, 2,5, +∞) |
| Z-TB/U | pendek / normal / tinggi | sama dengan Z-BB/U |
| Skor risiko (0–100) | rendah / sedang / tinggi | segitiga (0, 0, 30) / segitiga (25, 45, 65) / segitiga (60, 100, 100) |

Rumus fungsi keanggotaan segitiga $(a,b,c)$ dan trapesium $(a,b,c,d)$:

$$\mu_{segitiga}(x)=\max\left(\min\left(\frac{x-a}{b-a},\ \frac{c-x}{c-b}\right),\ 0\right)$$

$$\mu_{trapesium}(x)=\max\left(\min\left(\frac{x-a}{b-a},\ 1,\ \frac{d-x}{d-c}\right),\ 0\right)$$

Pada bahu kiri atau kanan ($a=b$ atau $c=d$), sisi lerengnya dianggap bernilai 1 sehingga Z-score yang sangat ekstrem tetap terklasifikasi penuh sebagai "rendah/pendek" atau "tinggi".

### Basis aturan
| Rule | Kondisi (IF) | Output (THEN) |
|---|---|---|
| R1 | BB rendah AND TB pendek | risiko tinggi |
| R2 | BB rendah AND TB normal | risiko tinggi |
| R3 | BB normal AND TB pendek | risiko tinggi |
| R4 | BB normal AND TB normal | risiko rendah |
| R5 | BB tinggi AND TB tinggi | risiko rendah |
| R6 | BB tinggi AND TB normal | risiko rendah |
| R7 | BB normal AND TB tinggi | risiko rendah |

Tidak ada rule yang menghasilkan himpunan output *sedang*; nilai sedang muncul hanya bila centroid jatuh di antara dua himpunan output.

### Tahapan inferensi Mamdani
1. **Fuzzifikasi:** setiap Z-score diubah menjadi derajat keanggotaan pada himpunan rendah/normal/tinggi.
2. **Kekuatan aktivasi rule (α):** operator AND memakai minimum, misalnya $\alpha_{R1}=\min(\mu_{BB\,rendah},\ \mu_{TB\,pendek})$.
3. **Implikasi:** himpunan output dipotong pada tinggi α: $\mu'_k(y)=\min(\alpha_k,\ \mu_{out}(y))$.
4. **Agregasi:** semua hasil implikasi digabung dengan maksimum, dengan $\alpha_{tinggi}=\max(\alpha_{R1},\alpha_{R2},\alpha_{R3})$ dan $\alpha_{rendah}=\max(\alpha_{R4},\dots,\alpha_{R7})$.
5. **Defuzzifikasi (centroid):**

$$y^{*}=\frac{\sum_{y} y\cdot\mu_{agg}(y)}{\sum_{y}\mu_{agg}(y)}$$

6. **Kategori skor:** ≤ 30 risiko rendah; 31–65 risiko sedang; > 65 risiko tinggi.

### Hasil untuk B03
Dengan $Z_{BB/U}=-2{,}92$ dan $Z_{TB/U}=-2{,}59$, keduanya berada di bahu "rendah/pendek" sehingga $\mu=1$ dan $\alpha_{R1}=\min(1,1)=1$; rule lain bernilai 0. Output yang terpotong pada tinggi 1 adalah segitiga (60, 100, 100) utuh, yang centroid-nya $\frac{60+100+100}{3}=86{,}67$. Skor 86,67 > 65 sehingga B03 berkategori **risiko tinggi**.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ---------- Fungsi keanggotaan (satu definisi saja) ----------
def trimf(x, a, b, c):
    x = np.asarray(x, dtype=float)
    kiri = np.ones_like(x) if a == b else (x - a) / (b - a)
    kanan = np.ones_like(x) if b == c else (c - x) / (c - b)
    return np.clip(np.minimum(kiri, kanan), 0, 1)

def trapmf(x, a, b, c, d):
    x = np.asarray(x, dtype=float)
    naik = np.ones_like(x) if a == b else (x - a) / (b - a)
    turun = np.ones_like(x) if c == d else (d - x) / (d - c)
    return np.clip(np.minimum(np.minimum(naik, 1), turun), 0, 1)

# Domain dan himpunan output skor risiko
universe = np.arange(0, 100.01, 0.1)
out_rendah = trimf(universe, 0, 0, 30)
out_sedang = trimf(universe, 25, 45, 65)   # tidak dipakai langsung oleh rule
out_tinggi = trimf(universe, 60, 100, 100)

def fuzzifikasi_z(z):
    """Keanggotaan Z-score (dipakai untuk BB/U maupun TB/U). Bahu kiri/kanan
    dibuat 'tak hingga' (+-10) agar Z ekstrem tetap terklasifikasi."""
    return {
        "rendah": float(trapmf(z, -10, -10, -2.5, -1.5)),
        "normal": float(trimf(z, -2.5, 0, 2.5)),
        "tinggi": float(trapmf(z, 1.5, 2.5, 10, 10)),
    }

def fis_mamdani(z_bbu, z_tbu):
    """Mengembalikan (skor_centroid, alpha_rule, kurva_agregasi)."""
    bb, tb = fuzzifikasi_z(z_bbu), fuzzifikasi_z(z_tbu)
    alpha = {
        "R1": min(bb["rendah"], tb["rendah"]),
        "R2": min(bb["rendah"], tb["normal"]),
        "R3": min(bb["normal"], tb["rendah"]),
        "R4": min(bb["normal"], tb["normal"]),
        "R5": min(bb["tinggi"], tb["tinggi"]),
        "R6": min(bb["tinggi"], tb["normal"]),
        "R7": min(bb["normal"], tb["tinggi"]),
    }
    a_tinggi = max(alpha["R1"], alpha["R2"], alpha["R3"])
    a_rendah = max(alpha["R4"], alpha["R5"], alpha["R6"], alpha["R7"])
    agregasi = np.maximum(np.minimum(a_rendah, out_rendah),
                          np.minimum(a_tinggi, out_tinggi))
    skor = np.nan if agregasi.sum() == 0 else float(np.sum(universe * agregasi) / agregasi.sum())
    return skor, alpha, agregasi

def kategori_risiko(skor):
    if pd.isna(skor):   return "Tidak Terklasifikasi"
    if skor <= 30:      return "Risiko Rendah"
    if skor <= 65:      return "Risiko Sedang"
    return "Risiko Tinggi"

# ---------- Contoh B03: estimasi soal vs estimasi kohort ----------
z_soal = ((7.4 - 10.9) / 1.2, (74 - 82.3) / 3.2)
b03 = df_a[df_a["ID"] == "B03"].iloc[0]
z_kohort = (b03["Z_BB_U"], b03["Z_TB_U"])

versi = {"Estimasi soal (median/SD contoh)": z_soal, "Estimasi kohort (Dataset A)": z_kohort}
baris, kurva = [], {}
for nama, (zb, zt) in versi.items():
    skor, alpha, agr = fis_mamdani(zb, zt)
    kurva[nama] = (skor, agr)
    baris.append({"Versi Z-score": nama, "Z_BB/U": round(zb, 2), "Z_TB/U": round(zt, 2),
                  **{k: round(v, 3) for k, v in alpha.items()},
                  "Skor FIS": round(skor, 2), "Kategori": kategori_risiko(skor)})
display(pd.DataFrame(baris).set_index("Versi Z-score"))

fig, axes = plt.subplots(1, 2, figsize=(11, 3.8), sharey=True)
for ax, (nama, (skor, agr)) in zip(axes, kurva.items()):
    ax.fill_between(universe, agr, alpha=0.35)
    ax.plot(universe, agr)
    ax.axvline(skor, linestyle="--", color="red", label=f"Centroid = {skor:.2f}")
    ax.set_title(nama, fontsize=10); ax.set_xlabel("Skor risiko"); ax.grid(True); ax.legend()
axes[0].set_ylabel("Derajat keanggotaan")
fig.suptitle("Agregasi Mamdani & centroid - Balita B03", y=1.02)
plt.tight_layout(); plt.show()

### Penjelasan alpha-cut dan centroid

**Alpha-cut (clipping).** Setiap rule menghasilkan kekuatan aktivasi α, yaitu nilai minimum derajat keanggotaan seluruh kondisinya. Himpunan output rule tersebut dipotong mendatar pada ketinggian α:

$$\mu'_k(y)=\min\left(\alpha_k,\ \mu_{out,k}(y)\right)$$

Rule yang kondisinya hanya terpenuhi sebagian (α kecil) memberi pengaruh kecil, sedangkan rule yang terpenuhi penuh (α = 1) memberi pengaruh penuh.

**Agregasi dan centroid.** Semua himpunan terpotong digabung dengan operator maksimum, lalu satu nilai tegas diperoleh dari titik berat (centroid) daerah hasil agregasi:

$$y^{*}=\frac{\sum_{y} y\cdot\mu_{agg}(y)}{\sum_{y}\mu_{agg}(y)}$$

Pada program, penjumlahan dilakukan pada domain 0–100 dengan langkah 0,1. Sel berikut menampilkan tabel α setiap rule dan kurva agregasi (dengan garis centroid) untuk B03.

### Menjalankan FIS pada seluruh Dataset A dan membandingkannya dengan `Status_gizi`

In [ ]:
# Kategori status gizi menurut Z-score (BB/U dan TB/U)
def kategori_bbu(z):
    if pd.isna(z): return "Tidak diketahui"
    if z < -3:     return "Berat Badan Sangat Kurang"
    if z < -2:     return "Berat Badan Kurang"
    if z <= 2:     return "Normal"
    return "Berat Badan Lebih"

def kategori_tbu(z):
    if pd.isna(z): return "Tidak diketahui"
    if z < -3:     return "Sangat Pendek"
    if z < -2:     return "Pendek"
    if z <= 3:     return "Normal"
    return "Tinggi"

df_a["Kategori_BB_U"] = df_a["Z_BB_U"].apply(kategori_bbu)
df_a["Kategori_TB_U"] = df_a["Z_TB_U"].apply(kategori_tbu)
display(df_a["Kategori_BB_U"].value_counts().rename_axis("Kategori BB/U").reset_index(name="Jumlah"))
display(df_a["Kategori_TB_U"].value_counts().rename_axis("Kategori TB/U").reset_index(name="Jumlah"))

# FIS untuk seluruh Dataset A
df_a["Skor_Risiko_FIS"] = df_a.apply(lambda r: fis_mamdani(r["Z_BB_U"], r["Z_TB_U"])[0], axis=1)
df_a["Kategori_Risiko_FIS"] = df_a["Skor_Risiko_FIS"].apply(kategori_risiko)

print("FIS berhasil dihitung:", df_a["Skor_Risiko_FIS"].notna().sum(), "dari", len(df_a))
display(df_a["Kategori_Risiko_FIS"].value_counts().rename_axis("Kategori").reset_index(name="Jumlah Balita"))

print("Perbandingan kategori FIS vs Status_gizi (label dataset):")
display(pd.crosstab(df_a["Status_gizi"], df_a["Kategori_Risiko_FIS"], margins=True))
print("Rata-rata skor FIS per Status_gizi:")
display(df_a.groupby("Status_gizi")["Skor_Risiko_FIS"].agg(["mean", "median", "min", "max"]).round(2))

## Soal 1b — Sistem Pakar berbasis aturan

Basis aturan hanya memakai fakta yang tersedia pada Dataset A: Z-BB/U, Z-TB/U, LILA, dan skor FIS. Berat lahir tidak tersedia sehingga tidak dijadikan syarat rule.

| Rule | Kondisi (IF) | Kesimpulan (THEN) |
|---|---|---|
| R9 | Z-TB/U < −2 AND LILA < 12,5 AND skor FIS > 65 | STUNTING |
| R8 | Z-TB/U < −2 AND skor FIS > 65 | STUNTING |
| R7 | Z-TB/U < −2 AND Z-BB/U < −2 | STUNTING |
| R4 | Z-BB/U < −2 AND Z-TB/U ≥ −2 | GIZI KURANG |
| R5 | 11,5 ≤ LILA ≤ 12,4 AND Z-TB/U ≥ −2 | GIZI KURANG |
| R6 | 31 ≤ skor FIS ≤ 65 AND Z-TB/U ≥ −2 | GIZI KURANG |
| R1 | Z-BB/U ≥ −2 AND Z-TB/U ≥ −2 AND LILA ≥ 12,5 AND skor FIS ≤ 30 | NORMAL |
| R2 | −2 ≤ Z-BB/U ≤ 2 AND −2 ≤ Z-TB/U ≤ 2 AND LILA ≥ 12,5 | NORMAL |
| R3 | Z-BB/U ≥ −2 AND Z-TB/U ≥ −2 AND skor FIS ≤ 30 | NORMAL |

### Resolusi konflik rule
Sebuah balita dapat memenuhi lebih dari satu rule. Konflik diselesaikan dengan strategi **rule pertama yang cocok berdasarkan urutan prioritas**:
1. **Prioritas kelas:** STUNTING → GIZI KURANG → NORMAL. Kondisi yang paling perlu perhatian dievaluasi lebih dahulu agar tidak tertutup oleh rule normal.
2. **Dalam kelas yang sama**, rule dengan syarat lebih banyak (lebih spesifik) dievaluasi lebih dahulu, yaitu R9 → R8 → R7 pada kelas stuntung dan R1 → R2 → R3 pada kelas normal.
3. Rule yang cocok pertama menentukan hasil dan saran tindak lanjut, dan evaluasi berhenti.

Bila tidak ada rule yang terpenuhi, hasilnya adalah **PERLU PEMERIKSAAN LANJUT**, tidak dipaksa menjadi NORMAL. Sistem ini adalah demonstrasi rule base untuk pembelajaran, bukan diagnosis klinis.

In [ ]:
def sistem_pakar(row):
    """Mengembalikan (hasil, rule_aktif, saran) untuk satu balita."""
    bb, tb = row["Z_BB_U"], row["Z_TB_U"]
    lila, fis = row["LILA_cm"], row["Skor_Risiko_FIS"]

    if tb < -2 and lila < 12.5 and fis > 65:
        return "STUNTING", "R9", "Prioritaskan pemeriksaan tenaga kesehatan, evaluasi asupan gizi, dan susun rencana tindak lanjut."
    if tb < -2 and fis > 65:
        return "STUNTING", "R8", "Lakukan pemeriksaan pertumbuhan lanjutan, evaluasi riwayat kesehatan, dan pantau tinggi badan."
    if tb < -2 and bb < -2:
        return "STUNTING", "R7", "Ukur ulang antropometri dan konsultasikan ke dokter atau petugas gizi."
    if bb < -2 and tb >= -2:
        return "GIZI KURANG", "R4", "Tingkatkan asupan bergizi (terutama protein hewani) dan pantau berat badan rutin."
    if 11.5 <= lila <= 12.4 and tb >= -2:
        return "GIZI KURANG", "R5", "Konsultasi petugas gizi dan evaluasi asupan nutrisi."
    if 31 <= fis <= 65 and tb >= -2:
        return "GIZI KURANG", "R6", "Periksa status gizi, edukasi orang tua, dan pantau pertumbuhan berkala."
    if bb >= -2 and tb >= -2 and lila >= 12.5 and fis <= 30:
        return "NORMAL", "R1", "Pertahankan pola makan bergizi seimbang dan pantau rutin di Posyandu."
    if -2 <= bb <= 2 and -2 <= tb <= 2 and lila >= 12.5:
        return "NORMAL", "R2", "Lanjutkan makanan bergizi seimbang, imunisasi lengkap, dan pemeriksaan berkala."
    if bb >= -2 and tb >= -2 and fis <= 30:
        return "NORMAL", "R3", "Pertahankan pola asuh dan pola makan yang baik serta pantau BB dan TB."
    return "PERLU PEMERIKSAAN LANJUT", "Tidak ada aturan", "Tidak ada rule yang cocok; lakukan pemeriksaan lanjut oleh tenaga kesehatan."

def jalankan_pakar(df):
    hasil = df.apply(sistem_pakar, axis=1, result_type="expand")
    return hasil[0], hasil[1], hasil[2]

df_a["Hasil_Sistem_Pakar"], df_a["Aturan_Aktif"], df_a["Saran"] = jalankan_pakar(df_a)

display(df_a["Hasil_Sistem_Pakar"].value_counts().rename_axis("Hasil").reset_index(name="Jumlah Balita"))
display(df_a["Aturan_Aktif"].value_counts().rename_axis("Aturan").reset_index(name="Jumlah Aktivasi"))

print("Balita yang tidak tercakup rule apa pun:")
display(df_a.loc[df_a["Aturan_Aktif"] == "Tidak ada aturan",
                 ["ID", "Nama", "Z_BB_U", "Z_TB_U", "LILA_cm", "Skor_Risiko_FIS", "Status_gizi"]])

print("Hasil Sistem Pakar vs Status_gizi (label dataset):")
status_up = df_a["Status_gizi"].str.upper()
display(pd.crosstab(status_up, df_a["Hasil_Sistem_Pakar"], margins=True))
print("Akurasi 3 kelas (Sistem Pakar vs label):", round((status_up == df_a["Hasil_Sistem_Pakar"]).mean(), 3))

### Contoh penelusuran B02 (Siti Rahayu)
Data: umur 36 bulan, BB 11,8 kg, TB 88 cm, LILA 12,1 cm, ASI eksklusif Tidak, label dataset Gizi Kurang. Sel berikut menjalankan rule base pada B02 dan menampilkan rule yang aktif. Penelusuran dilakukan berurutan; rule yang kondisinya tidak terpenuhi dilewati sampai ditemukan rule pertama yang cocok.

In [ ]:
b02 = df_a[df_a["ID"] == "B02"].iloc[0]
print(f"B02 | Z_BB/U={b02['Z_BB_U']:.2f}  Z_TB/U={b02['Z_TB_U']:.2f}  LILA={b02['LILA_cm']}  Skor FIS={b02['Skor_Risiko_FIS']:.2f}")
hasil, rule, saran = sistem_pakar(b02)
print(f"Rule aktif : {rule}")
print(f"Kesimpulan : {hasil}")
print(f"Saran      : {saran}")
print(f"Label dataset: {b02['Status_gizi']}")

## Soal 1c — XGBoost, Decision Tree, FIS, dan Sistem Pakar

### Desain evaluasi
- **Target:** `Label_risiko` (0 = Normal, 1 = Risiko). **Fitur:** umur, BB, TB, LILA, jenis kelamin, dan ASI eksklusif; `ID` dan `Nama` tidak dipakai.
- **Split:** 80% latih dan 20% uji, stratified, `random_state=42`. Data uji tidak dipakai selama tuning.
- **Tuning** memakai Stratified 5-Fold CV pada data latih dengan metrik F1:

| Model | Hyperparameter yang dituning | Jumlah kombinasi |
|---|---|---|
| XGBoost | `n_estimators` {100, 200}, `learning_rate` {0,03, 0,1}, `max_depth` {2, 3, 4}, `subsample` {0,8, 1,0}, `colsample_bytree` {0,8, 1,0} | 48 |
| Decision Tree | `max_depth` {2–6}, `min_samples_leaf` {1, 3, 5, 10}, `criterion` {gini, entropy} | 40 |

- **FIS dan Sistem Pakar** tidak dilatih, sehingga cukup dievaluasi pada baris data uji yang sama.

### Konversi ke kelas biner
- **FIS:** kelas 1 (risiko) bila skor ≥ 60, mengikuti ambang contoh pada Lampiran Data Pendukung bagian 5; nilai ini juga sejajar dengan batas bawah himpunan output *tinggi* ([60, 100]). Bila tidak ada rule yang aktif, skor dianggap 100 (berisiko). Sebagai analisis sensitivitas ditampilkan pula ambang yang dipilih hanya dari data latih.
- **Sistem Pakar:** NORMAL → 0; GIZI KURANG dan STUNTING → 1. Hasil "perlu pemeriksaan lanjut" (bila ada) dianggap 1 sebagai pendekatan konservatif untuk skrining.

### Metrik evaluasi
Dengan TP, TN, FP, FN dari confusion matrix (kelas positif = risiko):

$$\text{Accuracy}=\frac{TP+TN}{TP+TN+FP+FN} \qquad \text{Precision}=\frac{TP}{TP+FP} \qquad \text{Recall}=\frac{TP}{TP+FN}$$

$$F1=\frac{2\cdot \text{Precision}\cdot \text{Recall}}{\text{Precision}+\text{Recall}}$$

ROC-AUC adalah luas di bawah kurva TPR terhadap FPR, atau peluang bahwa balita berisiko mendapat skor lebih tinggi daripada balita normal. ROC-AUC Sistem Pakar tidak dihitung (NaN) karena keluarannya kelas tegas, bukan skor atau probabilitas.

In [ ]:
# Z-score Dataset B (diperlukan oleh FIS dan Sistem Pakar; BUKAN fitur XGBoost).
# Subjek Dataset B sama dengan Dataset A, sehingga referensi kohort yang sama dipakai.
print("ID dan urutan Dataset A = Dataset B:", bool((df_a["ID"].to_numpy() == df_b["ID"].to_numpy()).all()))
df_b["Z_BB_U"], df_b["Z_TB_U"] = hitung_z_kohort(df_b, ref_kohort)
print("Z_BB_U kosong:", df_b["Z_BB_U"].isna().sum(), "| Z_TB_U kosong:", df_b["Z_TB_U"].isna().sum())

In [ ]:
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, confusion_matrix, ConfusionMatrixDisplay, RocCurveDisplay)
from sklearn.tree import DecisionTreeClassifier

# Target biner dari teks "0 (Normal)" / "1 (Risiko)"
y = df_b["Label_risiko"].astype(str).str.extract(r"^\s*(\d)")[0].astype(int)
display(pd.crosstab(df_b["Label_risiko"], y, rownames=["Label asli"], colnames=["Target biner"]))

feature_cols = ["Umur_bulan", "BB_kg", "TB_cm", "LILA_cm", "Jenis_kelamin", "ASI_eksklusif_6_bulan"]
numeric_features = ["Umur_bulan", "BB_kg", "TB_cm", "LILA_cm"]
categorical_features = ["Jenis_kelamin", "ASI_eksklusif_6_bulan"]
assert not {"ID", "Nama"} & set(feature_cols), "ID/Nama tidak boleh menjadi fitur!"

X = df_b[feature_cols].copy()
preprocess = ColumnTransformer([
    ("num", "passthrough", numeric_features),
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, stratify=y, random_state=42)
print("Train:", X_train.shape, "| Test:", X_test.shape)
print("Proporsi kelas train:", y_train.value_counts(normalize=True).round(3).to_dict())
print("Proporsi kelas test :", y_test.value_counts(normalize=True).round(3).to_dict())

In [ ]:
# ---------- Tuning XGBoost (5 hyperparameter, Stratified 5-Fold CV, hanya data latih) ----------
from xgboost import XGBClassifier

xgb_pipeline = Pipeline([
    ("prep", preprocess),
    ("model", XGBClassifier(objective="binary:logistic", eval_metric="logloss",
                            random_state=42, n_jobs=-1)),
])
param_grid_xgb = {
    "model__n_estimators": [100, 200],
    "model__learning_rate": [0.03, 0.1],
    "model__max_depth": [2, 3, 4],
    "model__subsample": [0.8, 1.0],
    "model__colsample_bytree": [0.8, 1.0],
}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
grid_xgb = GridSearchCV(xgb_pipeline, param_grid_xgb, scoring="f1", cv=cv, n_jobs=-1, refit=True)
grid_xgb.fit(X_train, y_train)
best_xgb = grid_xgb.best_estimator_
print("Hyperparameter terbaik XGBoost:", grid_xgb.best_params_)
print("F1 CV terbaik:", round(grid_xgb.best_score_, 4))

# ---------- Decision Tree (dituning dengan prosedur yang sama agar perbandingan adil) ----------
dt_pipeline = Pipeline([
    ("prep", preprocess),
    ("model", DecisionTreeClassifier(random_state=42)),
])
param_grid_dt = {
    "model__max_depth": [2, 3, 4, 5, 6],
    "model__min_samples_leaf": [1, 3, 5, 10],
    "model__criterion": ["gini", "entropy"],
}
grid_dt = GridSearchCV(dt_pipeline, param_grid_dt, scoring="f1", cv=cv, n_jobs=-1, refit=True)
grid_dt.fit(X_train, y_train)
best_dt = grid_dt.best_estimator_
print("\nHyperparameter terbaik Decision Tree:", grid_dt.best_params_)
print("F1 CV terbaik:", round(grid_dt.best_score_, 4))

In [ ]:
# ---------- FIS dan Sistem Pakar pada Dataset B ----------
df_b["Skor_Risiko_FIS"] = df_b.apply(lambda r: fis_mamdani(r["Z_BB_U"], r["Z_TB_U"])[0], axis=1)
df_b["Skor_FIS_eval"] = df_b["Skor_Risiko_FIS"].fillna(100.0)   # tanpa rule aktif -> dianggap berisiko
df_b["Hasil_Sistem_Pakar"], df_b["Aturan_Aktif"], _ = jalankan_pakar(df_b)

# Ambang FIS utama = 60 (contoh pada lampiran dosen); ambang alternatif dicari HANYA dari data latih
idx_train, idx_test = X_train.index, X_test.index
skor_train = df_b.loc[idx_train, "Skor_FIS_eval"]
kandidat = np.unique(skor_train.round(2))
f1_kandidat = [f1_score(y_train, (skor_train > t).astype(int), zero_division=0) for t in kandidat]
cutoff_tuned = float(kandidat[int(np.argmax(f1_kandidat))])
print(f"Ambang FIS utama = 60 (lampiran dosen) | ambang hasil tuning pada data latih = {cutoff_tuned:.2f}")

pred_fis = (df_b.loc[idx_test, "Skor_FIS_eval"] >= 60).astype(int)
pred_fis_tuned = (df_b.loc[idx_test, "Skor_FIS_eval"] > cutoff_tuned).astype(int)
pred_pakar = (df_b.loc[idx_test, "Hasil_Sistem_Pakar"] != "NORMAL").astype(int)
skor_fis_test = df_b.loc[idx_test, "Skor_FIS_eval"]

In [ ]:
# ---------- Evaluasi semua metode pada test set yang sama ----------
def metrik(nama, y_true, pred, skor=None):
    return {
        "Model": nama,
        "Accuracy": accuracy_score(y_true, pred),
        "Precision": precision_score(y_true, pred, zero_division=0),
        "Recall": recall_score(y_true, pred, zero_division=0),
        "F1-score": f1_score(y_true, pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_true, skor) if skor is not None else np.nan,
    }

pred_xgb = best_xgb.predict(X_test);  proba_xgb = best_xgb.predict_proba(X_test)[:, 1]
pred_dt = best_dt.predict(X_test);    proba_dt = best_dt.predict_proba(X_test)[:, 1]

res_df = pd.DataFrame([
    metrik("XGBoost (tuned)", y_test, pred_xgb, proba_xgb),
    metrik("Decision Tree (tuned)", y_test, pred_dt, proba_dt),
    metrik("FIS Mamdani (skor >= 60)", y_test, pred_fis, skor_fis_test),
    metrik(f"FIS Mamdani (cutoff {cutoff_tuned:.1f}, tuning di train)", y_test, pred_fis_tuned, skor_fis_test),
    metrik("Sistem Pakar", y_test, pred_pakar, None),
]).set_index("Model")
print("Hasil evaluasi pada test set (n =", len(y_test), "):")
display(res_df.round(4))
print("ROC-AUC Sistem Pakar = NaN karena keluarannya kelas crisp, bukan skor/probabilitas.")

In [ ]:
# ---------- Confusion matrix, kurva ROC, dan feature importance ----------
preds = {
    "XGBoost": pred_xgb, "Decision Tree": pred_dt,
    "FIS (skor >= 60)": pred_fis, "FIS (cutoff tuned)": pred_fis_tuned, "Sistem Pakar": pred_pakar,
}
fig, axes = plt.subplots(1, 5, figsize=(19, 3.6))
for ax, (nama, p) in zip(axes, preds.items()):
    ConfusionMatrixDisplay(confusion_matrix(y_test, p, labels=[0, 1]),
                           display_labels=["Normal", "Risiko"]).plot(ax=ax, colorbar=False)
    ax.set_title(nama, fontsize=10)
plt.tight_layout(); plt.show()

fig, ax = plt.subplots(figsize=(5.5, 4.5))
for nama, s in {"XGBoost": proba_xgb, "Decision Tree": proba_dt, "FIS (skor)": skor_fis_test}.items():
    RocCurveDisplay.from_predictions(y_test, s, name=nama, ax=ax)
ax.plot([0, 1], [0, 1], "k--", alpha=0.4)
ax.set_title("Kurva ROC (test set)"); ax.grid(True); plt.show()

nama_fitur = best_xgb.named_steps["prep"].get_feature_names_out()
imp = pd.Series(best_xgb.named_steps["model"].feature_importances_, index=nama_fitur).sort_values()
imp.plot.barh(figsize=(6, 3.5), title="Feature importance XGBoost"); plt.tight_layout(); plt.show()

In [ ]:
# Ringkasan otomatis dari tabel hasil (angka mengikuti hasil eksekusi, bukan ditulis manual)
unggul = lambda kol: res_df[kol].idxmax()
print("Accuracy tertinggi :", unggul("Accuracy"), f"({res_df['Accuracy'].max():.3f})")
print("Recall tertinggi   :", unggul("Recall"), f"({res_df['Recall'].max():.3f})")
print("F1-score tertinggi :", unggul("F1-score"), f"({res_df['F1-score'].max():.3f})")
print("ROC-AUC tertinggi  :", unggul("ROC-AUC"), f"({res_df['ROC-AUC'].max():.3f})")
fn = {nama: int(((y_test == 1) & (p == 0)).sum()) for nama, p in preds.items()}
print("Kasus berisiko yang terlewat (false negative) per metode:", fn)

### Interpretasi dan keputusan untuk Puskesmas

Ringkasan angka (model terbaik pada tiap metrik dan jumlah kasus berisiko yang terlewat) ditampilkan oleh sel di atas. Dasar pertimbangan untuk memilih metode adalah sebagai berikut.

**1. Metrik yang diprioritaskan.** Pada skrining risiko stunting, recall kelas Risiko lebih penting daripada accuracy. Balita berisiko yang terlewat (*false negative*) terlambat ditangani, sedangkan *false positive* hanya menambah pemeriksaan lanjutan. Precision dan F1-score tetap diperhatikan agar beban pemeriksaan tidak berlebihan.

**2. Keandalan hasil.** Data uji hanya 40 baris, sehingga selisih kecil antar model dapat berasal dari satu atau dua sampel. Karena itu F1-score validasi silang pada data latih ikut dipertimbangkan, dan kesimpulan dinyatakan sebagai indikasi, bukan kepastian.

**3. Perbedaan cara kerja.** XGBoost dan Decision Tree belajar langsung dari label pada data latih. FIS dan Sistem Pakar tidak dilatih: keduanya bekerja dari Z-score estimasi dan ambang rule yang dirancang manual, sehingga kinerjanya bergantung pada kecocokan rule dengan data.

**4. Kemudahan penjelasan.**
- Sistem Pakar paling mudah diaudit karena setiap keputusan dapat ditelusuri ke rule tertentu dan disertai saran tindak lanjut.
- FIS memberi skor kontinu sehingga tingkat risiko bisa diurutkan, tetapi perlu penjelasan fungsi keanggotaan.
- Decision Tree menjadi jalan tengah: akurat dan aturannya masih dapat dibaca.
- XGBoost cenderung paling kuat menangkap pola tidak linear, tetapi paling sulit dijelaskan; feature importance membantu menunjukkan fitur yang berpengaruh.

**5. Rekomendasi.** Model dengan recall dan F1-score terbaik pada tabel hasil dipakai sebagai alat bantu skrining awal. Sistem Pakar dipakai untuk menjelaskan alasan dan memberi saran tindak lanjut kepada petugas dan orang tua. Setiap hasil tetap diverifikasi petugas kesehatan, dan hasil dari data simulasi ini tidak boleh dipakai sebagai diagnosis.

## Soal 2 — Forward Chaining dan Backward Chaining (kasus Hana)

### Basis aturan tetap dari soal
- R1: BBLow AND umur < 24 bulan → risiko_stunting_dini.
- R2: risiko_stunting_dini AND ASI_eksklusif_tidak → stunting_berpotensi_parah.
- R3: tinggi_badan_pendek AND berat_badan_kurang → gizi_buruk.
- R4: gizi_buruk OR stunting_berpotensi_parah → rujuk_puskesmas.
- R5: rujuk_puskesmas AND orang_tua_menolak = TIDAK → jadwalkan_pemantauan_intensif = YA.

Fakta yang diketahui dalam penelusuran kasus Hana mengikuti soal: BBLow=YA, umur<24=YA, ASI eksklusif tidak=YA, tinggi_badan_pendek=TIDAK, berat_badan_kurang=YA, orang_tua_menolak=TIDAK.

In [ ]:
# Forward chaining: menambahkan fakta baru sampai goal ditemukan atau tidak ada rule aktif.
facts_fc = {
    "BBLow": True,
    "umur_kurang_24": True,
    "ASI_eksklusif_tidak": True,
    "tinggi_badan_pendek": False,
    "berat_badan_kurang": True,
    "orang_tua_menolak": False
}
trace_fc = []

if facts_fc["BBLow"] and facts_fc["umur_kurang_24"]:
    facts_fc["risiko_stunting_dini"] = True
    trace_fc.append("R1 aktif: risiko_stunting_dini = YA")

if facts_fc.get("risiko_stunting_dini", False) and facts_fc["ASI_eksklusif_tidak"]:
    facts_fc["stunting_berpotensi_parah"] = True
    trace_fc.append("R2 aktif: stunting_berpotensi_parah = YA")

if facts_fc["tinggi_badan_pendek"] and facts_fc["berat_badan_kurang"]:
    facts_fc["gizi_buruk"] = True
    trace_fc.append("R3 aktif: gizi_buruk = YA")
else:
    trace_fc.append("R3 tidak aktif: tinggi_badan_pendek = TIDAK")

if facts_fc.get("gizi_buruk", False) or facts_fc.get("stunting_berpotensi_parah", False):
    facts_fc["rujuk_puskesmas"] = True
    trace_fc.append("R4 aktif: rujuk_puskesmas = YA")

if facts_fc.get("rujuk_puskesmas", False) and not facts_fc["orang_tua_menolak"]:
    facts_fc["jadwalkan_pemantauan_intensif"] = True
    trace_fc.append("R5 aktif: jadwalkan_pemantauan_intensif = YA")

print("\n".join(trace_fc))
print("Goal:", facts_fc.get("jadwalkan_pemantauan_intensif", False))

### Backward chaining
Backward chaining dimulai dari goal `jadwalkan_pemantauan_intensif=YA`, lalu mencari rule yang menghasilkan goal (R5). R5 memerlukan `rujuk_puskesmas=YA` dan orang tua tidak menolak. Untuk membuktikan rujukan, sistem memeriksa R4; R4 dapat dipenuhi oleh `stunting_berpotensi_parah=YA` atau `gizi_buruk=YA`. Cabang R2 berhasil melalui R1 dan fakta ASI eksklusif tidak. Cabang R3 gagal karena `tinggi_badan_pendek=TIDAK`, walaupun `berat_badan_kurang=YA`. Karena cabang R2 cukup untuk memenuhi OR pada R4, goal tetap terbukti.

In [ ]:
# Backward chaining: mesin inferensi sungguhan (rekursif, goal -> subgoal), bukan sekadar teks
fakta = {"BBLow": True, "umur_kurang_24": True, "ASI_eksklusif_tidak": True,
         "tinggi_badan_pendek": False, "berat_badan_kurang": True, "orang_tua_menolak": False}

# Format: rule -> (operator, [(nama_fakta, nilai_yang_dibutuhkan), ...], kesimpulan)
aturan = {
    "R1": ("AND", [("BBLow", True), ("umur_kurang_24", True)], "risiko_stunting_dini"),
    "R2": ("AND", [("risiko_stunting_dini", True), ("ASI_eksklusif_tidak", True)], "stunting_berpotensi_parah"),
    "R3": ("AND", [("tinggi_badan_pendek", True), ("berat_badan_kurang", True)], "gizi_buruk"),
    "R4": ("OR",  [("gizi_buruk", True), ("stunting_berpotensi_parah", True)], "rujuk_puskesmas"),
    "R5": ("AND", [("rujuk_puskesmas", True), ("orang_tua_menolak", False)], "jadwalkan_pemantauan_intensif"),
}
YA = lambda v: "YA" if v else "TIDAK"
trace_bc = []

def buktikan(nama, nilai=True, depth=0):
    pad = "   " * depth
    if nama in fakta:                                    # fakta yang sudah diketahui
        ok = fakta[nama] == nilai
        trace_bc.append(f"{pad}fakta {nama} = {YA(fakta[nama])} (dibutuhkan {YA(nilai)}) -> {'cocok' if ok else 'TIDAK cocok'}")
        return ok
    penghasil = [r for r, (_, _, h) in aturan.items() if h == nama]
    if not penghasil:                                    # bukan fakta & tidak ada rule penghasil
        trace_bc.append(f"{pad}{nama}: tidak ada fakta/rule -> dianggap TIDAK")
        return nilai is False
    if nilai is False:                                   # closed-world: tidak terbukti = TIDAK
        return not buktikan(nama, True, depth)
    for r in penghasil:
        op, kondisi, _ = aturan[r]
        trace_bc.append(f"{pad}Subgoal {nama}: coba {r} ({op})")
        hasil = None
        for n, v in kondisi:
            ok = buktikan(n, v, depth + 1)
            if op == "AND" and not ok: hasil = False; break
            if op == "OR" and ok:      hasil = True;  break
        if hasil is None:
            hasil = (op == "AND")                         # semua AND lolos / semua OR gagal
        trace_bc.append(f"{pad}-> {r} {'BERHASIL' if hasil else 'GAGAL'}")
        if hasil:
            return True
    return False

goal = "jadwalkan_pemantauan_intensif"
berhasil = buktikan(goal, True)
print(f"Goal: {goal} = YA ?\n")
print("\n".join(trace_bc))
print(f"\nHasil backward chaining: goal {'TERBUKTI' if berhasil else 'TIDAK terbukti'}")

### Perbandingan efisiensi
Kedua metode mencapai kesimpulan yang sama, yaitu pemantauan intensif dijadwalkan. Backward chaining lebih terarah karena hanya menelusuri rule yang mendukung goal, tetapi tetap perlu memeriksa cabang alternatif pada OR. Forward chaining memulai dari seluruh fakta dan menembakkan rule yang antecedent-nya terpenuhi; pendekatan ini cocok ketika sistem perlu menghasilkan banyak kesimpulan dari fakta yang tersedia. Pada kasus Hana, backward chaining lebih efisien untuk satu goal spesifik, sedangkan forward chaining lebih sesuai untuk pemantauan semua konsekuensi yang mungkin.

## Catatan batasan dan validitas

1. **Data simulasi.** Dataset bersifat sintetis untuk pembelajaran, bukan data klinis, dan hasilnya tidak dapat digeneralisasi ke kondisi nyata. Seluruh analisis hanya memakai Dataset A dan B.
2. **Z-score estimasi.** Z-score contoh B03 memakai median/SD contoh dari soal; Z-score 200 balita memakai median/SD kohort Dataset A per jenis kelamin dan kelompok umur. Keduanya adalah estimasi pembelajaran, bukan Z-score WHO. Akibatnya status "rendah" atau "pendek" bersifat relatif terhadap kohort, dan SD kohort yang besar membuat Z-score cenderung tampak lebih kecil dari yang akan diberikan standar WHO.
3. **Kelompok kecil.** Median dan SD pada kelompok umur dengan sedikit balita kurang stabil, dan posisi pengukuran (telentang atau berdiri) tidak dikoreksi karena tidak tersedia.
4. **Rule dan fungsi keanggotaan.** Rule FIS dan Sistem Pakar adalah rancangan pembelajaran, bukan standar klinis. Ambang FIS ≥ 60 mengikuti contoh pada lampiran, dan hasilnya dapat berubah bila ambang atau fungsi keanggotaan diubah (lihat analisis sensitivitas ambang).
5. **Evaluasi.** Data uji kecil (40 baris), sehingga metrik bersifat indikatif dan dapat berubah bila split berbeda.
6. **Penggunaan.** Hasil seluruh metode adalah alat bantu pembelajaran dan tidak boleh dipakai sebagai diagnosis; keputusan klinis tetap pada tenaga kesehatan.